In [1]:
# ===============================
# 1. Setup and Installations
# ===============================
!nvidia-smi
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr evaluate accelerate -q
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

import torch
import pandas as pd
import nltk
import os
from tqdm import tqdm
from datasets import load_dataset, load_from_disk
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline
)
import evaluate

nltk.download("punkt")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ===============================
# 2. Load Model and Tokenizer
# ===============================
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

# ===============================
# 3. Load Dataset (with fallback)
# ===============================
try:
    # Try to load directly from Hugging Face
    dataset_samsum = load_dataset("samsum")
    print("✅ Loaded SAMSum dataset from Hugging Face Hub")
except Exception as e:
    print(f"⚠️ Could not load from Hugging Face: {e}")
    print("Attempting to load from local zip...")

    # Download and unzip the dataset
    !wget -q https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
    !unzip -q -o summarizer-data.zip

    # List extracted contents to help debug
    print("\nExtracted files:")
    !ls -la

    # Find the directory containing .arrow files (the dataset)
    dataset_path = None
    for root, dirs, files in os.walk('.'):
        if any(f.endswith('.arrow') for f in files):
            dataset_path = root
            break

    if dataset_path is None:
        raise FileNotFoundError(
            "Could not find dataset in extracted files. "
            "Please check the zip contents manually using '!unzip -l summarizer-data.zip'."
        )

    print(f"Loading dataset from {dataset_path}")
    dataset_samsum = load_from_disk(dataset_path)
    print("✅ Loaded SAMSum dataset from local disk")

print(dataset_samsum)

split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]
print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")

# Display a sample
print("\nDialogue:")
print(dataset_samsum["test"][1]["dialogue"])
print("\nSummary:")
print(dataset_samsum["test"][1]["summary"])

# ===============================
# 4. Preprocessing (Tokenization)
# ===============================
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(
        example_batch['dialogue'],
        max_length=1024,
        truncation=True,
        padding="max_length"
    )

    with tokenizer.as_target_tokenizer():
        target_encodings = tokenizer(
            example_batch['summary'],
            max_length=128,
            truncation=True,
            padding="max_length"
        )

    return {
        'input_ids': input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels': target_encodings['input_ids']
    }

dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features, batched=True)
print(dataset_samsum_pt["train"])

# ===============================
# 5. Data Collator
# ===============================
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ===============================
# 6. Training Arguments and Trainer
# ===============================
training_args = Seq2SeqTrainingArguments(
    output_dir='pegasus-samsum',
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=2,      # adjust based on GPU memory
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    gradient_accumulation_steps=8,
    predict_with_generate=True,
    generation_max_length=128,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=dataset_samsum_pt["train"],
    eval_dataset=dataset_samsum_pt["validation"],
)

# ===============================
# 7. Training
# ===============================
trainer.train()

# ===============================
# 8. Evaluation with ROUGE
# ===============================
def generate_batch_sized_chunks(list_of_elements, batch_size):
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                               batch_size=16, device=device,
                               column_text="dialogue",
                               column_summary="summary"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(zip(article_batches, target_batches), total=len(article_batches)):
        inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                           padding="max_length", return_tensors="pt").to(device)

        summaries = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            length_penalty=0.8,
            num_beams=8,
            max_length=128
        )

        decoded_summaries = [
            tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            for s in summaries
        ]

        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    return metric.compute()

rouge_metric = evaluate.load('rouge')
score = calculate_metric_on_test_ds(
    dataset_samsum['test'][0:10],
    rouge_metric,
    trainer.model,
    tokenizer,
    batch_size=2,
    column_text='dialogue',
    column_summary='summary'
)

rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = {rn: score[rn].mid.fmeasure for rn in rouge_names}
rouge_df = pd.DataFrame([rouge_dict], index=['pegasus'])
print("\nROUGE Scores:")
print(rouge_df)

# ===============================
# 9. Save Model and Tokenizer
# ===============================
model.save_pretrained("pegasus-samsum-model")
tokenizer.save_pretrained("pegasus-samsum-tokenizer")

# ===============================
# 10. Test with Pipeline
# ===============================
tokenizer = AutoTokenizer.from_pretrained("pegasus-samsum-tokenizer")
model = AutoModelForSeq2SeqLM.from_pretrained("pegasus-samsum-model").to(device)

pipe = pipeline("summarization", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

sample_text = dataset_samsum["test"][0]["dialogue"]
reference = dataset_samsum["test"][0]["summary"]

gen_kwargs = {"length_penalty": 0.8, "num_beams": 8, "max_length": 128}

print("\nDialogue:")
print(sample_text)
print("\nReference Summary:")
print(reference)
print("\nModel Summary:")
print(pipe(sample_text, **gen_kwargs)[0]["summary_text"])

Sun Mar 22 16:50:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.99                 Driver Version: 555.99         CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   49C    P8              7W /   50W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Nikhil\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Using device: cuda


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

'wget' is not recognized as an internal or external command,
operable program or batch file.



Extracted files:


unzip:  cannot find or open summarizer-data.zip, summarizer-data.zip.zip or summarizer-data.zip.ZIP.


total 112
drwxr-xr-x 1 Nikhil 197121     0 Mar 22 16:05 .
drwxr-xr-x 1 Nikhil 197121     0 Mar  7 14:55 ..
-rw-r--r-- 1 Nikhil 197121  8351 Mar  7 14:58 01_data_ingestion.ipynb
-rw-r--r-- 1 Nikhil 197121  6731 Mar  8 22:00 02_data_validation.ipynb
-rw-r--r-- 1 Nikhil 197121 10042 Mar 10 17:08 03_data_transformation.ipynb
-rw-r--r-- 1 Nikhil 197121 21490 Mar 16 16:42 04_model_trainer.ipynb
-rw-r--r-- 1 Nikhil 197121 11400 Mar 16 17:21 05_model_evaluation.ipynb
drwxr-xr-x 1 Nikhil 197121     0 Mar 10 17:01 logs
drwxr-xr-x 1 Nikhil 197121     0 Mar 13 17:37 model_trainer
-rw-r--r-- 1 Nikhil 197121 34144 Mar 22 16:51 text_summarization.ipynb


FileNotFoundError: Could not find dataset in extracted files. Please check the zip contents manually using '!unzip -l summarizer-data.zip'.